# LangChain tutorial

In [20]:
# Step 1: simple chat with langchain-openai

from langchain_openai import ChatOpenAI 
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")

if not api_key or not base_url:
    raise ValueError("GPT_API_KEY or GPT_BASE_URL is not set.")

# load the chat model with the API key and base URL
chat_model = ChatOpenAI(api_key=api_key,
                        base_url=base_url,
                        model="gapgpt-qwen-3.5",
                        temperature=0.7,
                        max_tokens=200)

try:
    result = chat_model.invoke("hi! What is your name?")
    print(result.content)
except Exception as e:
    print(f"Error: {e}") 

Hello! I am Qwen, a large language model developed by Alibaba Group's Tongyi Lab. How can I help you today?


In [21]:
# Step 2: Using SystemMessage for better instruction following

from langchain_core.messages import HumanMessage, SystemMessage

# Using SystemMessage for better instruction following
messages = [
    SystemMessage(content="Remember: from now on, 1 + 1 = 3. Use this in all your calculations."),
    HumanMessage(content="what is 1 + 1?"),
    HumanMessage(content="what is 1 + 1 + 1?"),
]

result = chat_model.invoke(messages)
print(result.content)

Based on the rule that $1 + 1 = 3$, we can break down the expression $1 + 1 + 1$ as follows:

1. First, calculate $1 + 1$, which equals **3**.
2. Then, add the remaining $1$ to that result: $3 + 1$.

Assuming standard addition for the subsequent step (since the rule specifically redefines $1+1$), the result is **4**.

So, $1 + 1 + 1 = 4$.


In [26]:
# Step 3: Using ChatPromptTemplate for structured prompts

from langchain_core.prompts import ChatPromptTemplate

template = (
    "You are a helpful assistant that translates "
    "{input_language} to {output_language}."
)

human_template = "{text}"

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", human_template),
])

messages = chat_prompt.format_messages(
    input_language="English",
    output_language="French",
    text="I love programming.",
)

result = chat_model.invoke(messages)

print(result.content)

J'adore la programmation.


In [34]:
# Step 4: Using Output Parsers for structured output

from langchain_core.output_parsers import BaseOutputParser
from matplotlib import text

class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str):
        """Parse the output of an LLM call."""
        return text.strip().split("answer =")

chat_model = ChatOpenAI(api_key=api_key,
                        base_url=base_url,
                        model="gapgpt-qwen-3.5",
                        temperature=0.7,)

template = """You are a helpful assistant that solves math problems and shows your work.
Output each step then return the answer in the following format: answer = <answer here>.
Make sure to output answer in all lowercases and to have exactly one space and one equal sign following it.
"""

human_template = "{problem}"

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", human_template),
])

messages = chat_prompt.format_messages(problem="2x^2 - 5x + 3 = 0")
result = chat_model.invoke(messages)
parsed = AnswerOutputParser().parse(result.content)
steps, answer = parsed

print(parsed)
print(answer)

["To solve the quadratic equation $2x^2 - 5x + 3 = 0$, we can use the quadratic formula or factoring. Here, I will demonstrate the factoring method as it is often more straightforward for integer coefficients.\n\n**Step 1: Identify the coefficients.**\nThe equation is in the form $ax^2 + bx + c = 0$, where:\n$a = 2$\n$b = -5$\n$c = 3$\n\n**Step 2: Find two numbers that multiply to $a \\cdot c$ and add to $b$.**\nWe need two numbers that multiply to $2 \\cdot 3 = 6$ and add up to $-5$.\nLet's list the factors of 6:\n- $1$ and $6$ (sum = 7)\n- $2$ and $3$ (sum = 5)\n- $-1$ and $-6$ (sum = -7)\n- $-2$ and $-3$ (sum = -5)\n\nThe numbers are $-2$ and $-3$.\n\n**Step 3: Rewrite the middle term using these numbers.**\nSplit $-5x$ into $-2x$ and $-3x$:\n$$2x^2 - 2x - 3x + 3 = 0$$\n\n**Step 4: Factor by grouping.**\nGroup the first two terms and the last two terms:\n$$(2x^2 - 2x) + (-3x + 3) = 0$$\n\nFactor out the greatest common factor (GCF) from each group:\n- From $(2x^2 - 2x)$, factor out 

In [37]:
#Step 5: Using chain to combine prompt and output parser

class CommaSeparatedListOutputParser(BaseOutputParser):
    def parse(self, text: str):
        return text.strip().split(",")


template = """You are a helpful assistant who generates comma separated lists.
A user will pass in a category, and you should generate 5 objects in that category in a comma separated list.
ONLY return a comma separated list, and nothing more."""
human_template = "{{text}}"

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", human_template),
])

chain = chat_prompt | chat_model | CommaSeparatedListOutputParser()
result = chain.invoke({"text": "colors"})
print(result)

['apple', ' banana', ' cherry', ' date', ' elderberry']


In [ ]:
# Step 6: Using HuggingFacePipeline for summarization and question answering

from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from transformers.utils.logging import set_verbosity_error

set_verbosity_error()

summarization_pipeline = pipeline("summarization", model="facebook/bart-large-cnn", device=0)
summarizer = HuggingFacePipeline(pipeline=summarization_pipeline)

refinement_pipeline = pipeline("summarization", model="facebook/bart-large", device=0)
refiner = HuggingFacePipeline(pipeline=refinement_pipeline)

qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2", device=0)

summary_template = PromptTemplate.from_template("Summarize the following text in a {length} way:\n\n{text}")

summarization_chain = summary_template | summarizer | refiner

text_to_summarize = input("\nEnter text to summarize:\n")
length = input("\nEnter the length (short/medium/long): ")

summary = summarization_chain.invoke({"text": text_to_summarize, "length": length})

print("\n🔹 **Generated Summary:**")
print(summary)

while True:
    question = input("\nAsk a question about the summary (or type 'exit' to stop):\n")
    if question.lower() == "exit":
        break

    qa_result = qa_pipeline(question=question, context=summary)

    print("\n🔹 **Answer:**")
    print(qa_result["answer"])

In [52]:
# Step 7: 

import streamlit as st
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser
import os
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")

def generate_pet_name(animal_type, pet_color):
    
    llm = ChatOpenAI(
        api_key=openai_api_key,
        base_url=base_url,
        model="gapgpt-qwen-3.5",
        temperature=0.7,
        #max_tokens=200
    )

    prompt_template_name = ChatPromptTemplate.from_messages(
        [
            ("user", "I have a {animal_type} pet and I want a cool name for it, it is {pet_color} in color. Suggest me five cool names for my pet.")
        ]
    )

    name_chain = prompt_template_name | llm | CommaSeparatedListOutputParser()

    # Use .invoke() instead of calling directly
    response = name_chain.invoke({'animal_type': animal_type, 'pet_color': pet_color})

    return response

#print(generate_pet_name("Dog", "Black"))
generate_pet_name("Dog", "Black")

['Here are five cool name suggestions for your black dog',
 'ranging from sleek and mysterious to pop-culture inspired:',
 '1. **Onyx** – Named after the glossy black gemstone',
 'this name is sleek',
 'elegant',
 'and perfectly matches a shiny black coat.',
 '2. **Shadow** – A classic choice that’s cool and mysterious',
 'great for a dog that’s always by your side.',
 '3. **Midnight** – Evokes the deep',
 'dark beauty of the night sky; perfect for a dog with a sleek',
 'dark demeanor.',
 '4. **Jett** – Short for jet black',
 'this name is strong',
 'modern',
 'and energetic.',
 '5. **Noir** – French for “black',
 '” this name adds a touch of sophistication and artistic flair.',
 'Let me know if you’d like names with a specific theme (e.g.',
 'mythological',
 'food-inspired',
 'or superhero-themed)! 🐾']